# Monthly Sales Forecasting — Academic Prototype

**NOVA IMS | Business Cases with Data Science | March 2025**

This is an output-redacted, data-free *reference copy* of an academic time-series forecasting workflow. Company data, actual financial findings, forecasts, and source-specific identifiers have been intentionally omitted. The code requires authorized input data and adaptation before execution. This notebook has **not** been validated for production use. See README for methodological limitations.

**Academic group:** Jude Gbenimako, Tomás Ribeiro, Elcano Gaspar, Afonso Silva, Rui Reis.

## Contents
1. Libraries and data inputs
2. Exploratory analysis and preprocessing
3. Feature engineering
4. Time-series cross-validation and XGBoost modeling
5. Forecasting workflow

### <font color = '#13b0a6'>**Introduction**</font>

### Project objective and scope
This academic exercise investigates whether historical sales and external market indicators can support monthly sales forecasts. The workflow explores aggregation, data cleaning, seasonal decomposition, lag and rolling-window features, Bayesian tuning of XGBoost, and time-based validation.

**Confidentiality:** Original company data and business-specific results are omitted. Input files are not included. The financial figures and predictions previously embedded in output cells are not present.

**Technical caveat:** Historical code is retained for study but requires authorized data, input-schema adaptation, and additional validation; reported company forecasting performance is not claimed.

## <font color = '#13b0a6'>  **1 Libraries, Functions and Datasets** </font> <a class="anchor" id="libraries-functions-and-datasets"></a>

#### <font color = '#13b0a6'> **1.1. Import the Needed Libraries** </font> <a class="anchor" id="import-the-needed-libraries"></a>

In [ ]:
# System & Warnings
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_absolute_percentage_error

# Tables
from tabulate import tabulate

from forecast_helpers import *

# Style
color_pal = sns.color_palette()
plt.style.use('fivethirtyeight')
sns.set()

In [ ]:
# Source-specific market-indicator codes are intentionally omitted.
# If you have authorized data, map its columns to the normalized feature
# names required by the later analysis cells.
column_name_mapping = {}


#### <font color = '#13b0a6'> **1.2. Data import** </font> <a class="anchor" id="import-the-dataset"></a>

In [ ]:
# Load datasets from local files that are explicitly excluded from Git.
# These files are not distributed in the repository.
from pathlib import Path
sales_data_path = Path("data/sales.csv")
market_data_path = Path("data/market_indicators.xlsx")
if not sales_data_path.exists() or not market_data_path.exists():
    raise FileNotFoundError(
        "Authorized data not supplied. Place compatible sales and market files in data/ "
        "and update the input schema if necessary."
    )
sales_data = pd.read_csv(sales_data_path, sep=";", parse_dates=["DATE"], dayfirst=True, index_col="DATE")
market_data = pd.read_excel(market_data_path, skiprows=2)
market_data.rename(columns={market_data.columns[0]: "Date"}, inplace=True)
market_data["Date"] = pd.to_datetime(market_data["Date"].str[:4] + "-" + market_data["Date"].str[5:] + "-01", errors="coerce")
market_data.dropna(subset=["Date"], inplace=True)
market_data.set_index("Date", inplace=True)
print("Sales columns:", sales_data.columns.to_list())
print("Market columns:", market_data.columns.to_list())


### <font color = '#13b0a6'>  **2. Exploratory Data Analysis** </font> <a class="anchor" id="exploratory-data-analysis-eda"></a> 

#### <font color = '#13b0a6'> **2.1. Exploration of Data** </font><a class="anchor" id="data-exploration"></a>

#### <font color = '#13b0a6'> **2.1.1. Sales Data** </font>

In [ ]:
# Check Data Information
print("Sales Data Info:")
print(tabulate(sales_data.info(), headers='keys', tablefmt='pretty'))

In [ ]:
non_numeric_values = sales_data[~sales_data['Sales_EUR'].apply(lambda x: str(x).replace('.', '', 1).isdigit())]['Sales_EUR']
print(non_numeric_values.unique())

In [ ]:
sales_data['Sales_EUR'] = sales_data['Sales_EUR'].str.replace(',', '.')

# Convert the 'Sales_EUR' column to float
sales_data['Sales_EUR'] = pd.to_numeric(sales_data['Sales_EUR'], errors='coerce')

# Check Data Information
print("Sales Data Info:")
print(tabulate(sales_data.info(), headers='keys', tablefmt='pretty'))

In [ ]:
# Access the minimum and maximum dates from the index
min_date = sales_data.index.min()
max_date = sales_data.index.max()

print("Minimum Date:", min_date)
print("Maximum Date:", max_date)

Sales columns may need type conversion before aggregation. Verify data types using your authorized data.

#### <font color = '#13b0a6'> **2.1.2. Market Data** </font>

In [ ]:
# Check Data Information
print("Market Data Info:")
print(tabulate(market_data.info(), headers='keys', tablefmt='pretty'))

In [ ]:
# Replace column names using the mapping
market_data.columns = [column_name_mapping.get(col, col) for col in market_data.columns]

# Drop duplicate features
market_data = market_data.loc[:, ~market_data.columns.duplicated()]

# Display updated market_data columns
print("\n Updated Market Data Columns:")
print(market_data.columns)

# Display first few rows of updated market_data
print("\n First 5 Rows of Updated Market Data:")
print(market_data.head())

In [ ]:
# Access the minimum and maximum dates from the index
min_date = market_data.index.min()
max_date = market_data.index.max()

print("Minimum Date:", min_date)
print("Maximum Date:", max_date)

In [ ]:
# Display first few rows to analyze structure
print("First 5 Rows of Sales Data:")
print(tabulate(sales_data.head(), headers='keys', tablefmt='pretty'))

print("First 5 Rows of Sales Data:")
print(tabulate(market_data.head(), headers='keys', tablefmt='pretty'))

Check date coverage, column types, and missingness before joining sales and market indicators.

#### <font color = '#13b0a6'> **2.2. Univariate Analysis** </font>

#### <font color = '#13b0a6'> **2.2.1. Sales Data** </font>

In [ ]:
sales_data[sales_data['Sales_EUR']<0].index.year.value_counts(normalize=True)

In [ ]:
plot_sales_distribution(sales_data, product_column="Mapped_GCK", sales_column="Sales_EUR", custom_green="#13b0a6")

Investigate outliers and irregular sales activity. Business-specific observations were removed.

Let's aggregate the dataset to the product "Mapped_GCK " and "Sales_EUR" daily level so that we can have high volume metrics for demand forecasting throughout the country. The line of code below can also be done in SQL.
<i>select date,
        Mapped_GCK ,
        sum(Sales_EUR)
from sales_data
group by DATE, Mapped_GCK
</i>

In [ ]:
agg_sales = sales_data.groupby(['DATE','Mapped_GCK']).agg({'Sales_EUR':'sum'}).reset_index().sort_values(['Mapped_GCK','DATE'])

print("First 5 Rows of Aggregade Sales Data:")
print(tabulate(agg_sales.head(), headers='keys', tablefmt='pretty'))

In [ ]:
# Pivot the Data
total_sales_df = agg_sales.pivot(index='DATE', columns='Mapped_GCK', values='Sales_EUR')

# Fill NaN with Zeros were no sales for a specific combination of DATE and Product.
total_sales_df = total_sales_df.fillna(0)

# Print the Result
print("First 5 Rows of Total Sales Data:")
print(tabulate(total_sales_df.head(), headers='keys', tablefmt='pretty'))

In [ ]:
missing_data(total_sales_df)

In [ ]:
plot_total_sales_per_product(sales_data, date_col='DATE', product_col='Mapped_GCK', sales_col='Sales_EUR', title="Total Sales per Product")

Review the distribution of sales across product groups. Actual amounts and company-specific findings have been removed.

For modeling, the workflow compares product categories with different sales volumes. The public notebook uses dynamically computed groups rather than the original case-specific grouping.

### Category Grouping

In [ ]:
#lets break it down by thirds for low, mid, high
avg_daily_sales = total_sales_df.apply(np.mean, axis=0).sort_values()
low, mid = np.percentile(total_sales_df.apply(np.mean, axis=0).sort_values(),[33,66])

low_vol_columns = list(avg_daily_sales[avg_daily_sales<=low].index)
mid_vol_columns = avg_daily_sales[(avg_daily_sales>low) & (avg_daily_sales<mid)].index
high_vol_columns = avg_daily_sales[avg_daily_sales>=mid].index

Separate forecasting groups can help investigate how model performance differs by sales volume. Original company-specific results are omitted.

In [ ]:
#total_sales_df[low_vol_columns].plot.line()
plt.plot(total_sales_df[low_vol_columns])
plt.legend(low_vol_columns, loc='best') #, bbox_to_anchor=(1.1, 1.1))
plt.show()

Use this plot to investigate variability and intermittency in lower-volume series.

In [ ]:
plt.plot(total_sales_df[mid_vol_columns])
plt.legend(mid_vol_columns, loc='best', bbox_to_anchor=(1.1, 1.1))
plt.xticks(rotation=30)
plt.show()

Use this plot to investigate variability in mid-volume series.

In [ ]:
plt.plot(total_sales_df[high_vol_columns])
plt.legend(high_vol_columns, loc='best', bbox_to_anchor=(1.1, 1.1))
plt.xticks(rotation=30)
plt.show()

Use this plot to investigate patterns in higher-volume series.

Ploting time series components for each product group

In [ ]:
decompose_and_plot_sales(sales_data)

In [ ]:
decompose_and_plot_sales(sales_data)

Inspect the overlap between sales and market time series and study trend and seasonality using your own authorized inputs.

#### <font color = '#13b0a6'> **2.2.2. Market Data** </font>

In [ ]:
# Set dark background and larger figure
plt.figure(figsize=(18, 9))

# Focus countries - using consistent ordering
countries = ["Germany", "United States", "China", "Japan", "France", "United Kingdom", "Italy", "Switzerland", "Europe"]

for country in countries:
    # Plot Production (dashed line)
    plt.plot(
        market_data.index,
        market_data[f"{country}_PRO"],
        color=color_map[f"{country}_PRO"],
        linestyle="--",
        alpha=0.7 if country != "Germany" else 1,
        linewidth=0.5,
        label=f"{country} Production"
    )
    # Plot Shipments (solid line)
    plt.plot(
        market_data.index,
        market_data[f"{country}_SHP"],
        color=color_map[f"{country}_SHP"],
        linewidth=2 if country == "Germany" else 1.5,
        alpha=1 if country in ["Germany"] else 0.3,
        label=f"{country} Shipments"
    )

# Add horizontal line at 100 if data is index-based
plt.axhline(100, color='white', linestyle=':', alpha=0.4)

plt.title("Global Machinery & Electricals: Production vs. Shipments Trend",
          fontsize=18, pad=20, color='black')
plt.xlabel("Date", fontsize=12, color='black', labelpad=10)
plt.ylabel("Index Value (Volume)", fontsize=12, color='black', labelpad=10)
plt.legend(ncol=3, fontsize=10, framealpha=0.2)
plt.grid(alpha=0.15)
plt.tick_params(colors='black')
plt.tight_layout()
plt.show()

Market-indicator commentary based on the original project data has been removed.

Compare market indicators only where their licensing and provenance permit use.

In [ ]:
# Filter only Production Index columns (those originally from PRO_ columns)
production_cols = [
    col for col in market_data.columns
    if col.endswith("Machinery_PRO")
]

# Plot
plt.figure(figsize=(14, 6))
for col in production_cols:
    color = color_map.get(col, "#4CAF50")  # Use mapped color or default green
    alpha = 1 if col == "Germany_Machinery_PRO" else 0.3  # Highlight Germany only
    plt.plot(market_data.index, market_data[col],
             label=col.replace("_Machinery_PRO", ""),
             color=color,
             alpha=alpha,
             linewidth=2 if "Germany" in col else 1)

# Styling
plt.title("Production Index for Machinery and equipment n.e.c.", fontsize=14, color='black')
plt.xlabel("Date", color='black')
plt.ylabel("Index Value", color='black')
plt.legend(fontsize=9, framealpha=0.2)
plt.grid(alpha=0.2)
plt.tick_params(colors='black')
plt.tight_layout()
plt.show()

Market-indicator commentary based on the original project data has been removed.

Assess how external predictors could improve sales forecasts without drawing causal conclusions from correlation alone.

In [ ]:
for col in ["Base_Metals_Index", "Energy_Index", "Metals_Minerals_Index", "Natural_Gas_Index", "Crude_Oil_Index", "Copper_Index"]:
    plt.plot(market_data.index, market_data[col], label=col)

plt.xlabel("Year/Month")
plt.ylabel("Price Index")
plt.title("World Price")
plt.legend(loc="upper left", bbox_to_anchor=(1,1))  # Move legend outside plot
plt.xticks(rotation=45)
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
for col in ["US_Electrical_Equipment_Price", "UK_Electrical_Equipment_Price",
    "Italy_Electrical_Equipment_Price", "France_Electrical_Equipment_Price",
    "Germany_Electrical_Equipment_Price", "China_Electrical_Equipment_Price"]:
    plt.plot(market_data.index, market_data[col], label=col)

plt.xlabel("Year/Month")
plt.ylabel("Price Index")
plt.title("Electrical Equipment Price Index by Country")
plt.legend(loc="upper left", bbox_to_anchor=(1,1))  # Move legend outside plot
plt.xticks(rotation=45)
plt.grid(True)

plt.show()

#### <font color = '#13b0a6'> **2.3. Data preparation** </font>

In [ ]:
# Convert Daily Sales Data to Monthly Sales
# Ensure Sales_EUR is numeric first
sales_data["Sales_EUR"] = pd.to_numeric(sales_data["Sales_EUR"], errors="coerce")

# Group by Mapped_GCK and Month, summing sales
sales_monthly = sales_data.groupby(["Mapped_GCK", sales_data.index.to_period("M")])["Sales_EUR"].sum().reset_index()
sales_monthly = sales_monthly.rename(columns={"DATE": "Month"})  # Only if column is named DATE
sales_monthly["Month"] = sales_monthly["Month"].dt.to_timestamp()  # Convert period to datetime

# Verify the column names
print("\nColumn Names in Aggregated Sales Data:", sales_monthly.columns)
print(sales_monthly.head())

#### <font color = '#13b0a6'> **2.3.1 Data Cleaning** </font>

##### <font color = '#13b0a6'> **Outlier Removal** </font> <a class="anchor" id="fem"></a>

In [ ]:
interpolated_df_list = []

# Loop through each product
for product in sales_monthly["Mapped_GCK"].unique():
    df_product = sales_monthly[sales_monthly["Mapped_GCK"] == product][["Month", "Mapped_GCK", "Sales_EUR"]].copy()
    df_product = df_product.sort_values("Month").set_index("Month")

    # IQR to detect outliers
    Q1 = df_product["Sales_EUR"].quantile(0.25)
    Q3 = df_product["Sales_EUR"].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_mask = (df_product["Sales_EUR"] < lower_bound) | (df_product["Sales_EUR"] > upper_bound)
    outlier_count = outlier_mask.sum()

    print(f"Product {product} - Replacing {outlier_count} outlier rows out of {len(df_product)} (IQR Method)")

    # Replace outliers with NaN
    df_product.loc[outlier_mask, "Sales_EUR"] = np.nan

    # Interpolate missing (outlier) values linearly over time
    df_product["Sales_EUR"] = df_product["Sales_EUR"].interpolate(method="time")

    df_product["Mapped_GCK"] = product  # Restore column

    interpolated_df_list.append(df_product.reset_index())

# Final cleaned dataset with continuity preserved
sales_monthly = pd.concat(interpolated_df_list).sort_values(by=["Mapped_GCK", "Month"]).reset_index(drop=True)

print(f"\n Final imputed sales_monthly shape: {sales_monthly.shape}")

##### <font color = '#13b0a6'> **Handle Missing Values** </font> <a class="anchor" id="fem"></a>

In [ ]:
# Interpolate missing values per product in the MONTHLY data (not the original daily data)
sales_monthly["Sales_EUR"] = sales_monthly.groupby("Mapped_GCK")["Sales_EUR"].transform(
    lambda x: x.interpolate()
)

# Verify if Missing Values are Handled
print("\nMissing Values After Processing:")
print(sales_monthly.isna().sum())


In [ ]:
# Check Skewness
skewness = sales_monthly.groupby("Mapped_GCK")["Sales_EUR"].apply(lambda x: x.skew())
print("\nSkewness of Monthly Sales Data per Product:")
print(skewness)

Skewness summaries and underlying product-level statistics were removed. Recompute them using authorized data.

#### <font color = '#13b0a6'> **2.3.2 Stationarity** </font>

Check for Stationarity:

- Time series data should be stationary (constant mean and variance over time) for meaningful correlation analysis.

- Use the Augmented Dickey-Fuller (ADF) test to check for stationarity.

In [ ]:
# Collect ADF test results
adf_results = []

for product in sales_monthly["Mapped_GCK"].unique():
    product_sales = sales_monthly[sales_monthly["Mapped_GCK"] == product].sort_values("Month")
    result = check_stationarity(product_sales["Sales_EUR"], product)
    adf_results.append(result)

# Print results as table
print(tabulate(adf_results, headers="keys", tablefmt="pretty"))

#### <font color = '#13b0a6'> **2.3.3 Autocorrelation & Partial Autocorrelation** </font>

In [ ]:
# Create example product groups based on relative historical total sales,
# rather than including internal identifiers or original group membership.
ranked_products = (
    sales_monthly.groupby("Mapped_GCK")["Sales_EUR"]
    .sum().sort_values().index.to_numpy()
)
ranked_sections = np.array_split(ranked_products, 3)
product_groups = {
    "Group_A": ranked_sections[2].tolist(),  # higher volume
    "Group_B": ranked_sections[1].tolist(),  # middle volume
    "Group_C": ranked_sections[0].tolist(),  # lower volume
}
for group_name, products in product_groups.items():
    plot_acf_pacf_for_group(sales_monthly, group_name, products)


Autocorrelation and partial autocorrelation should be interpreted for each series using authorized data.

Check stationarity and autocorrelation as diagnostic information; investigate non-stationarity in forecast validation.

#### <font color = '#13b0a6'> **2.4. Feature Engineering** </font> <a class="anchor" id="fem"></a>

In [ ]:
# Step 2: Time-Based Features
sales_monthly["Month_Num"] = sales_monthly["Month"].dt.month
sales_monthly["Quarter"] = sales_monthly["Month"].dt.quarter
sales_monthly["Year"] = sales_monthly["Month"].dt.year

sales_monthly["IsMonthStart"] = sales_monthly["Month"].dt.is_month_start.astype(int)
sales_monthly["IsMonthEnd"] = sales_monthly["Month"].dt.is_month_end.astype(int)

group_a = product_groups["Group_A"]
group_b = product_groups["Group_B"]
group_c = product_groups["Group_C"]

target = "Sales_EUR"
target_log = "Sales_EUR_Log"

sales_monthly.set_index("Month", inplace=True)

# TimeSeriesSplit for CV
tss = TimeSeriesSplit(n_splits=5, test_size=10, gap=1)

#### <font color = '#13b0a6'> **Group A** </font> <a class="anchor" id="fem"></a>

In [ ]:
# Define Group A specific features
group_a_features = [
    "Sales_EUR",                # Original target
    "Month_Num", "Quarter", "Year",
    "Sales_Lag_1", "Sales_Lag_2", "Sales_Lag_3",
    "RollingMean_3", "RollingMean_6", "RollingStd_3",
    "Lag1_RollingMean3",
    "Mapped_GCK"
]

data_group_a = sales_monthly[sales_monthly["Mapped_GCK"].isin(group_a)].copy()

data_group_a = add_lags(data_group_a, target, [1, 2, 3])
data_group_a = add_rolling(data_group_a, target, [3, 6])

# ✅ Interaction Term
data_group_a["Lag1_RollingMean3"] = (
    data_group_a["Sales_Lag_1"] * data_group_a["RollingMean_3"]
)

# Final feature set
data_group_a = data_group_a[group_a_features]

# Set index as datetime
data_group_a.index = pd.to_datetime(data_group_a.index)
data_group_a = data_group_a.sort_index()

# Group A features update
group_a_features = [
    "Year", "Month_Num", "Quarter",
    "Sales_Lag_1", "Sales_Lag_2", "Sales_Lag_3",
    "RollingMean_3", "RollingMean_6", "RollingStd_3",
    "Lag1_RollingMean3"
]

data_group_a.head()

In [ ]:
missing_data(data_group_a)

#### <font color = '#13b0a6'> **Group B** </font> <a class="anchor" id="fem"></a>

In [ ]:
# Define Group B specific features
group_b_features = [
    "Sales_EUR",                # Original target
    "Month_Num", "Quarter", "Year",
    "Sales_Lag_3", "Sales_Lag_6","RollingStd_3", "Mapped_GCK"
]

# Create a copy with only Group B data
data_group_b = sales_monthly[sales_monthly["Mapped_GCK"].isin(group_b)].copy()

# Adding new features
data_group_b = add_lags(data_group_b, target, [3, 6])
data_group_b = add_rolling(data_group_b, target, [3, 6])

# Final feature set
data_group_b = data_group_b[group_b_features]

# Set index as datetime
data_group_b.index = pd.to_datetime(data_group_b.index)
data_group_b = data_group_b.sort_index()

# Group B features update
group_b_features = [
    "Year", "Month_Num", "Quarter",
    "Sales_Lag_3", "Sales_Lag_6", "RollingStd_3"
]

data_group_b.head()

In [ ]:
missing_data(data_group_b)

#### <font color = '#13b0a6'> **Group C** </font> <a class="anchor" id="fem"></a>

In [ ]:
# Define Group C specific features
group_c_features = [
    "Sales_EUR",
    "Month_Num", "Quarter", "Year",
    "Sales_Lag_6", "Sales_Lag_12","RollingMean_6", "Sales_EUR_Log", "Mapped_GCK"]

data_group_c = sales_monthly[sales_monthly["Mapped_GCK"].isin(group_c)].copy()

# Apply log transformation to Group C only
data_group_c["Sales_EUR_Log"] = data_group_c.apply(
    lambda row: np.log1p(row["Sales_EUR"]) if row["Sales_EUR"] > -1 else 0,
    axis=1
)

data_group_c = add_lags(data_group_c, target_log, [6, 12])
data_group_c = add_rolling(data_group_c, target_log, [6])

# Final feature set
data_group_c = data_group_c[group_c_features]

# Set index as datetime
data_group_c.index = pd.to_datetime(data_group_c.index)
data_group_c = data_group_c.sort_index()

# Group A features update
group_c_features = [
    "Month_Num", "Quarter", "Year",
    "Sales_Lag_6", "Sales_Lag_12","RollingMean_6"]

data_group_c

In [ ]:
missing_data(data_group_c)

### <font color = '#13b0a6'>  **3. Model Assessment** </font> <a class="anchor" id="model-assessment"></a>

**Modeling approach**
The accompanying original code implements XGBoost regressors with time-series cross-validation and Bayesian parameter tuning. Other models mentioned in the original academic planning narrative are not included in this notebook.

**Methodological caution:** Check for data leakage in feature engineering and ensure all transformations are fit inside each training fold before using results for decision-making.

#### <font color = '#13b0a6'> **3.1.  Cross Validation** </font><a class="anchor" id="cross-validation"></a>

#### <font color = '#13b0a6'> **Group A** </font> <a class="anchor" id="fem"></a>

In [ ]:
plot_time_series_folds(data_group_a, target, tss)

In [ ]:
xgb_bo_a = xgb_optimization(data_group_a, group_a_features, target, tss)

In [ ]:
# Extract best hyperparameters
best_params = xgb_bo_a.max["params"]

best_xgb_model_a = xgb.XGBRegressor(
    learning_rate=best_params["learning_rate"],
    max_depth=int(best_params["max_depth"]),
    n_estimators=int(best_params["n_estimators"]),
    gamma=best_params["gamma"],
    subsample=best_params["subsample"],
    colsample_bytree=best_params["colsample_bytree"],
    objective="reg:squarederror",
    verbosity=0
)

# Display best params
print("Best XGBoost Hyperparameters for Group A:")
for k, v in best_params.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

In [ ]:
results = xgb_evaluate_info_with_predictions(data_group_a, group_a_features, target, tss, best_params["learning_rate"],
                                             max_depth=int(best_params["max_depth"]), n_estimators=int(best_params["n_estimators"]),
                                             gamma=best_params["gamma"], subsample=best_params["subsample"], colsample_bytree=best_params["colsample_bytree"])

overfitting_plot(results)

Training for the whole dataset

In [ ]:
# Train on full Group A data
X = data_group_a[group_a_features]
y = data_group_a[target]

best_xgb_model_a.fit(X, y)

In [ ]:
# Evaluate per product
print("\n📌 RMSE per Product in Group A:")
for product in group_a:
    product_df = data_group_a[data_group_a["Mapped_GCK"] == product]

    X_prod = product_df[group_a_features]
    y_true = product_df[target]
    y_pred = best_xgb_model_a.predict(X_prod)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    print(f"\nProduct {product}: \n   RMSE = {rmse:.2f}")
    print(f"   MAPE: {mape:.2%}")

#### <font color = '#13b0a6'> **Group B** </font> <a class="anchor" id="fem"></a>

In [ ]:
plot_time_series_folds(data_group_b, target, tss)

In [ ]:
xgb_bo_b = xgb_optimization(data_group_b, group_b_features, target, tss)

In [ ]:
# Extract best hyperparameters
best_params = xgb_bo_b.max["params"]

best_xgb_model_b = xgb.XGBRegressor(
    learning_rate=best_params["learning_rate"],
    max_depth=int(best_params["max_depth"]),
    n_estimators=int(best_params["n_estimators"]),
    gamma=best_params["gamma"],
    subsample=best_params["subsample"],
    colsample_bytree=best_params["colsample_bytree"],
    objective="reg:squarederror",
    verbosity=0
)

# Display best params
print("Best XGBoost Hyperparameters for Group B:")
for k, v in best_params.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

In [ ]:
results = xgb_evaluate_info_with_predictions(data_group_b, group_b_features, target, tss, best_params["learning_rate"],
                                             max_depth=int(best_params["max_depth"]), n_estimators=int(best_params["n_estimators"]),
                                             gamma=best_params["gamma"], subsample=best_params["subsample"], colsample_bytree=best_params["colsample_bytree"])

overfitting_plot(results)

Training for the whole dataset

In [ ]:
# Train on full Group B data
X = data_group_b[group_b_features]
y = data_group_b[target]

best_xgb_model_b.fit(X, y)

In [ ]:
# Evaluate per product
print("\n📌 RMSE per Product in Group B:")
for product in group_b:
    product_df = data_group_b[data_group_b["Mapped_GCK"] == product]

    X_prod = product_df[group_b_features]
    y_true = product_df[target]
    y_pred = best_xgb_model_b.predict(X_prod)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    print(f"\nProduct {product}: \n   RMSE = {rmse:.2f}")
    print(f"   MAPE: {mape:.2%}")

#### <font color = '#13b0a6'> **Group C** </font> <a class="anchor" id="fem"></a>

In [ ]:
plot_time_series_folds(data_group_c, target_log, tss)

In [ ]:
tss = TimeSeriesSplit(n_splits=5, test_size=20, gap=1)

In [ ]:
xgb_bo_c = xgb_optimization(data_group_c, group_c_features, target_log, tss)

In [ ]:
# Extract best hyperparameters
best_params = xgb_bo_c.max["params"]

best_xgb_model_c = xgb.XGBRegressor(
    learning_rate=best_params["learning_rate"],
    max_depth=int(best_params["max_depth"]),
    n_estimators=int(best_params["n_estimators"]),
    gamma=best_params["gamma"],
    subsample=best_params["subsample"],
    colsample_bytree=best_params["colsample_bytree"],
    objective="reg:squarederror",
    verbosity=0
)

# Display best params
print("Best XGBoost Hyperparameters for Group C:")
for k, v in best_params.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

In [ ]:
# Extract best hyperparameters
best_params = xgb_bo_c.max["params"]

best_xgb_model_c = xgb.XGBRegressor(
    learning_rate=best_params["learning_rate"],
    max_depth=int(best_params["max_depth"]),
    n_estimators=int(best_params["n_estimators"]),
    gamma=best_params["gamma"],
    subsample=best_params["subsample"],
    colsample_bytree=best_params["colsample_bytree"],
    objective="reg:squarederror",
    verbosity=0
)

# Display best params
print("Best XGBoost Hyperparameters for Group B:")
for k, v in best_params.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

In [ ]:
results = xgb_evaluate_info_with_predictions(data_group_c, group_c_features, target_log, tss, best_params["learning_rate"],
                                             max_depth=int(best_params["max_depth"]), n_estimators=int(best_params["n_estimators"]),
                                             gamma=best_params["gamma"], subsample=best_params["subsample"], colsample_bytree=best_params["colsample_bytree"])

overfitting_plot(results)

Training for the whole dataset

In [ ]:
# Train on full Group C data
X = data_group_c[group_c_features]
y = data_group_c[target_log]

best_xgb_model_c.fit(X, y)

In [ ]:
# Evaluate per product
print("\n📌 RMSE per Product in Group C:")
for product in group_c:
    product_df = data_group_c[data_group_c["Mapped_GCK"] == product]

    X_prod = product_df[group_c_features]

    y_true = product_df[target]
    y_pred = np.expm1(best_xgb_model_c.predict(X_prod))

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    semape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\nProduct {product}: \n   RMSE = {rmse:.2f}")

## Experimental ten-month forecasts

The following code outlines the historical prototype. It is not a validated production forecasting pipeline. It must be re-tested with independent datasets before use.

#### <font color = '#13b0a6'> **Group A** </font> <a class="anchor" id="fem"></a>

In [ ]:
# Split into features/target
X_train = data_group_a[group_a_features]
y_train = data_group_a[target]

# Forecast 10 months ahead
future_forecasts = []

# Deep copy to simulate step-ahead prediction
forecast_df = data_group_a.copy()

# Loop per product in Group A
for product in group_a:
    product_df = forecast_df[forecast_df["Mapped_GCK"] == product].copy()

    for step in range(10):  # 10 months ahead
        last_row = product_df.iloc[-1].copy()
        forecast_date = last_row.name + pd.DateOffset(months=1)

        # Create new row with updated time features
        new_row = last_row.copy()
        new_row.name = forecast_date
        new_row["Year"] = forecast_date.year
        new_row["Month_Num"] = forecast_date.month
        new_row["Quarter"] = (forecast_date.month - 1) // 3 + 1
        new_row["IsMonthStart"] = int(forecast_date.is_month_start)
        new_row["IsMonthEnd"] = int(forecast_date.is_month_end)

        # Create lag features from previous values
        new_row["Sales_Lag_1"] = last_row["Sales_EUR"]
        new_row["Sales_Lag_2"] = last_row["Sales_Lag_1"]
        new_row["Sales_Lag_3"] = last_row["Sales_Lag_2"]

        # Update rolling mean/std (using last available rolling window)
        rolling_window = product_df.tail(3)["Sales_EUR"]
        new_row["RollingMean_3"] = rolling_window.mean()
        new_row["RollingStd_3"] = rolling_window.std() if len(rolling_window) > 1 else 0.0

        rolling_window6 = product_df.tail(6)["Sales_EUR"]
        new_row["RollingMean_6"] = rolling_window6.mean()

        # Interaction feature
        new_row["Lag1_RollingMean3"] = new_row["Sales_Lag_1"] * new_row["RollingMean_3"]

        # Predict sales
        X_pred = new_row[group_a_features].values.reshape(1, -1)
        predicted_sales = best_xgb_model_a.predict(X_pred)[0]
        new_row["Sales_EUR"] = predicted_sales

        # Append prediction
        product_df = pd.concat([product_df, pd.DataFrame([new_row])])
        future_forecasts.append({
            "Date": forecast_date,
            "Mapped_GCK": product,
            "Forecast_Sales_EUR": round(predicted_sales, 2)
        })

# Build Forecast DataFrame
forecast_result_df = pd.DataFrame(future_forecasts)
forecast_result_df_a = forecast_result_df.sort_values(["Mapped_GCK", "Date"])

# Display Forecast
print(" Forecast – Group A (May 2022 to Feb 2023):")
print(tabulate(forecast_result_df_a, headers="keys", tablefmt="pretty", showindex=False))

In [ ]:
# Define colors
hist_color = "#1f77b4"  # Blue for historical data
pred_color = "#d62728"  # Red for predicted data

forecast = forecast_result_df_a

# Loop through each unique product
for prod in forecast["Mapped_GCK"].unique():
    # Filter data for the current product
    ts_pred = forecast[forecast["Mapped_GCK"] == prod].copy()
    ts_pred.set_index("Date", inplace=True)

    ts_hist = sales_monthly[sales_monthly["Mapped_GCK"] == prod]["Sales_EUR"].copy()

    # Ensure the last historical point connects to the first predicted point
    if not ts_hist.empty and not ts_pred.empty:
        last_hist_point = ts_hist.iloc[-1]  # Last historical sales value
        first_pred_date = ts_pred.index.min()  # First predicted date

        # Create a small DataFrame with the last historical point
        transition_df = pd.DataFrame(
            {"Forecast_Sales_EUR": [last_hist_point]}, index=[first_pred_date]
        )

        # Concatenate the transition point to predicted data
        ts_pred = pd.concat([transition_df, ts_pred])

    # Plot
    plt.figure(figsize=(12, 6))

    plt.plot(ts_hist.index, ts_hist, label="Historical Sales", color=hist_color, linewidth=2)
    plt.plot(ts_pred.index, ts_pred["Forecast_Sales_EUR"], label="Predicted Sales",
             color=pred_color, linestyle="dashed", linewidth=2)

    # Formatting
    plt.title(f"Sales Evolution for Product {prod}", fontsize=14, fontweight='bold')
    plt.xlabel("Date", fontsize=12)
    plt.ylabel("Sales (€)", fontsize=12)
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)

    plt.savefig(f"fds {prod}", transparent=True, dpi=300, bbox_inches='tight')

    # Show plot
    plt.show()


#### <font color = '#13b0a6'> **Group B** </font> <a class="anchor" id="fem"></a>

In [ ]:
# Split into features/target
X_train = data_group_b[group_b_features]
y_train = data_group_b[target]

# ✅ Forecast 10 months ahead
future_forecasts = []

# Deep copy to simulate step-ahead prediction
forecast_df = data_group_b.copy()

# Loop per product in Group B
for product in group_b:
    product_df = forecast_df[forecast_df["Mapped_GCK"] == product].copy()

    for step in range(10):  # 10 months ahead
        last_row = product_df.iloc[-1].copy()
        forecast_date = last_row.name + pd.DateOffset(months=1)

        # Create new row with updated time features
        new_row = last_row.copy()
        new_row.name = forecast_date
        new_row["Year"] = forecast_date.year
        new_row["Month_Num"] = forecast_date.month
        new_row["Quarter"] = (forecast_date.month - 1) // 3 + 1
        new_row["IsMonthStart"] = int(forecast_date.is_month_start)
        new_row["IsMonthEnd"] = int(forecast_date.is_month_end)

        # Create lag features from previous values
        new_row["Sales_Lag_3"] = last_row["Sales_EUR"]
        new_row["Sales_Lag_6"] = last_row["Sales_Lag_3"]

        # Update rolling mean/std (using last available rolling window)
        rolling_window = product_df.tail(3)["Sales_EUR"]
        new_row["RollingStd_3"] = rolling_window.std() if len(rolling_window) > 1 else 0.0

        # Predict sales
        X_pred = new_row[group_b_features].values.reshape(1, -1)
        predicted_sales = best_xgb_model_b.predict(X_pred)[0]
        new_row["Sales_EUR"] = predicted_sales

        # Append prediction
        product_df = pd.concat([product_df, pd.DataFrame([new_row])])
        future_forecasts.append({
            "Date": forecast_date,
            "Mapped_GCK": product,
            "Forecast_Sales_EUR": round(predicted_sales, 2)
        })

# ✅ Build Forecast DataFrame
forecast_result_df = pd.DataFrame(future_forecasts)
forecast_result_df_b = forecast_result_df.sort_values(["Mapped_GCK", "Date"])

# Display Forecast
print(" Forecast – Group B (May 2022 to Feb 2023):")
print(tabulate(forecast_result_df_b, headers="keys", tablefmt="pretty", showindex=False))

In [ ]:
# Define colors
hist_color = "#1f77b4"  # Blue for historical data
pred_color = "#d62728"  # Red for predicted data

forecast = forecast_result_df_b

# Loop through each unique product
for prod in forecast["Mapped_GCK"].unique():
    # Filter data for the current product
    ts_pred = forecast[forecast["Mapped_GCK"] == prod].copy()
    ts_pred.set_index("Date", inplace=True)

    ts_hist = sales_monthly[sales_monthly["Mapped_GCK"] == prod]["Sales_EUR"].copy()

    # Ensure the last historical point connects to the first predicted point
    if not ts_hist.empty and not ts_pred.empty:
        last_hist_point = ts_hist.iloc[-1]  # Last historical sales value
        first_pred_date = ts_pred.index.min()  # First predicted date

        # Create a small DataFrame with the last historical point
        transition_df = pd.DataFrame(
            {"Forecast_Sales_EUR": [last_hist_point]}, index=[first_pred_date]
        )

        # Concatenate the transition point to predicted data
        ts_pred = pd.concat([transition_df, ts_pred])

    # Plot
    plt.figure(figsize=(12, 6))

    plt.plot(ts_hist.index, ts_hist, label="Historical Sales", color=hist_color, linewidth=2)
    plt.plot(ts_pred.index, ts_pred["Forecast_Sales_EUR"], label="Predicted Sales",
             color=pred_color, linestyle="dashed", linewidth=2)

    # Formatting
    plt.title(f"Sales Evolution for Product {prod}", fontsize=14, fontweight='bold')
    plt.xlabel("Date", fontsize=12)
    plt.ylabel("Sales (€)", fontsize=12)
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)

    plt.savefig(f"fds {prod}", transparent=True, dpi=300, bbox_inches='tight')

    # Show plot
    plt.show()

#### <font color = '#13b0a6'> **Group C** </font> <a class="anchor" id="fem"></a>

In [ ]:
# Split into features/target
X_train = data_group_c[group_c_features]
y_train = data_group_c[target_log]

# ✅ Forecast 10 months ahead
future_forecasts = []

# Deep copy to simulate step-ahead prediction
forecast_df = data_group_c.copy()

# Loop per product in Group C
for product in group_c:
    product_df = forecast_df[forecast_df["Mapped_GCK"] == product].copy()

    for step in range(10):  # 10 months ahead
        last_row = product_df.iloc[-1].copy()
        forecast_date = last_row.name + pd.DateOffset(months=1)

        # Create new row with updated time features
        new_row = last_row.copy()
        new_row.name = forecast_date
        new_row["Year"] = forecast_date.year
        new_row["Month_Num"] = forecast_date.month
        new_row["Quarter"] = (forecast_date.month - 1) // 3 + 1
        new_row["IsMonthStart"] = int(forecast_date.is_month_start)
        new_row["IsMonthEnd"] = int(forecast_date.is_month_end)

        # Create lag features from previous values
        new_row["Sales_Lag_6"] = last_row["Sales_EUR_Log"]
        new_row["Sales_Lag_12"] = last_row["Sales_Lag_6"]

        # Update rolling mean/std (using last available rolling window)
        rolling_window = product_df.tail(6)["Sales_EUR_Log"]
        new_row["RollingMean_6"] = rolling_window.mean()

        # Predict sales
        X_pred = new_row[group_c_features].values.reshape(1, -1)
        predicted_log_sales = best_xgb_model_c.predict(X_pred)[0]
        predicted_sales = np.expm1(predicted_log_sales)

        new_row["Sales_EUR_Log"] = predicted_sales

        # Append prediction
        product_df = pd.concat([product_df, pd.DataFrame([new_row])])
        future_forecasts.append({
            "Date": forecast_date,
            "Mapped_GCK": product,
            "Forecast_Sales_EUR": round(predicted_sales, 2)
        })

# Build Forecast DataFrame
forecast_result_df = pd.DataFrame(future_forecasts)
forecast_result_df_c = forecast_result_df.sort_values(["Mapped_GCK", "Date"])

# Display Forecast
print(" Forecast – Group C (May 2022 to Feb 2023):")
print(tabulate(forecast_result_df_c, headers="keys", tablefmt="pretty", showindex=False))

In [ ]:
# Define colors
hist_color = "#1f77b4"  # Blue for historical data
pred_color = "#d62728"  # Red for predicted data

forecast = forecast_result_df_c

# Loop through each unique product
for prod in forecast["Mapped_GCK"].unique():
    # Filter data for the current product
    ts_pred = forecast[forecast["Mapped_GCK"] == prod].copy()
    ts_pred.set_index("Date", inplace=True)

    ts_hist = sales_monthly[sales_monthly["Mapped_GCK"] == prod]["Sales_EUR"].copy()

    # Ensure the last historical point connects to the first predicted point
    if not ts_hist.empty and not ts_pred.empty:
        last_hist_point = ts_hist.iloc[-1]  # Last historical sales value
        first_pred_date = ts_pred.index.min()  # First predicted date

        # Create a small DataFrame with the last historical point
        transition_df = pd.DataFrame(
            {"Forecast_Sales_EUR": [last_hist_point]}, index=[first_pred_date]
        )

        # Concatenate the transition point to predicted data
        ts_pred = pd.concat([transition_df, ts_pred])

    # Plot
    plt.figure(figsize=(12, 6))

    plt.plot(ts_hist.index, ts_hist, label="Historical Sales", color=hist_color, linewidth=2)
    plt.plot(ts_pred.index, ts_pred["Forecast_Sales_EUR"], label="Predicted Sales",
             color=pred_color, linestyle="dashed", linewidth=2)

    # Formatting
    plt.title(f"Sales Evolution for Product {prod}", fontsize=14, fontweight='bold')
    plt.xlabel("Date", fontsize=12)
    plt.ylabel("Sales (€)", fontsize=12)
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)

    # Show plot
    plt.show()

#### <font color = '#13b0a6'> **3.3.  Forecast** </font>

In [ ]:
# Export experimental forecasts derived from authorized data ONLY.
# Avoid committing generated forecasts from confidential datasets.
from pathlib import Path
combined_forecast = pd.concat(
    [forecast_result_df_a, forecast_result_df_b, forecast_result_df_c],
    ignore_index=True
)
columns_to_export = ["Date", "Mapped_GCK", "Forecast_Sales_EUR"]
forecast_export = combined_forecast[columns_to_export].copy()
Path("outputs").mkdir(exist_ok=True)
forecast_export.to_csv("outputs/experimental_forecasts.csv", index=False)
print("Export saved locally in outputs/ (ignored by Git)")
